<h1>Workshop on Polygenic Scoring</h1>
<h3><b>Volos Summer School 2026</b></br></h3>
<i><h5>Ana Luiza Arruda<br>Konstantinos Hatzikotoulas</h5></i></br>



In this exercise, we will construct a polygenic score for Alzheimer's disease (AD) and apply it to our cohort with individual-level genotype data from the previous workshops.

**Step-by-step**
- Build a Polygenic Score (PGS) by applying LD clumping to a published and publicly available AD GWAS summary statistics
- Harmonise AD GWAS summary statistics with our target individual-level genotype data
- Apply the PGS to two populations of our data using PLINK
- Evaluate performance for a binary (AD) and a quantitative (cognitive function) trait
- Compare the transferability of PGS predictive power across populations

**Goal**: Understand why PGS behaves differently depending on trait architecture and ancestry.

During this practical we will use R and PLINK. Answers to questions are provided in the hidden cells. Please try to answer them yourself first.

To run the first cell please change the runtime type to Python 3. Then change it back to R. You can use `gc()` from time to time to free up memory.

## 1. Setup

In [ ]:
# ── Step 1a: Mount Google Drive ──────────────────────────────────
# Run this cell with a Python runtime (Runtime > Change runtime type > Python 3)
# then switch back to R for all subsequent cells
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Step 1b: Set working directory (R runtime) ───────────────────
setwd("/content/drive/My Drive/VSS_2026/PGS_workshop")

In [ ]:
# ── Step 1c: Install and load packages ───────────────────────────
install.packages(c("data.table", "ggplot2", "dplyr", "pROC"), quiet = TRUE)

library(data.table)
library(ggplot2)
library(dplyr)
library(pROC)

In [ ]:
# ── Step 1d: Download and install PLINK ──────────────────────────
cat(system("wget -q https://s3.amazonaws.com/plink1-assets/plink_linux_x86_64_20210606.zip", intern = TRUE), sep = "\n")
cat(system("unzip -o plink_linux_x86_64_20210606.zip", intern = TRUE), sep = "\n")
cat(system("rm -f prettify toy.* LICENSE", intern = TRUE), sep = "\n")
cat(system("./plink --version", intern = TRUE), sep = "\n")

## 2. Toy example: what is a PGS?

Before working with real data, let's build intuition with a minimal example.

A **Polygenic Score** is calculated as:

$$\text{PGS}_i = \sum_{j=1}^{M} g_{ij} \times \beta_j$$

Where:
- $g_{ij}$ = number of effect alleles for individual $i$ at SNP $j$ (0, 1, or 2)
- $\beta_j$ = effect size of SNP $j$ from a GWAS
- $M$ = number of SNPs included

The key idea is simple: **we weight each person's genotype by how much each variant is associated with the trait, then sum.**

In [ ]:
# Toy example: 10 SNPs, 5 individuals
set.seed(42)

n_snps <- 10
n_individuals <- 5

# Simulate GWAS effect sizes (betas)
toy_beta <- rnorm(n_snps, mean = 0, sd = 0.3)
cat("==== Simulated effect sizes (beta) ====\n")
print(round(toy_beta, 3))

# Simulate genotype matrix: rows = SNPs, cols = individuals
# Each cell = 0, 1, or 2 (copies of the effect allele)
toy_geno <- matrix(sample(0:2, n_snps * n_individuals, replace = TRUE),
                   nrow = n_snps, ncol = n_individuals)
cat("\n==== Simulated genotype matrix (SNPxindividuals) ====\n")
print(toy_geno)

In [ ]:
# Compute PGS: for each individual, sum(genotype * beta) across SNPs
toy_pgs <- colSums(toy_geno * toy_beta)
cat("\n==== PGS per individual ====\n"); print(round(toy_pgs, 3))

### Question
- What does a higher PGS value mean?
- Why do we need thousands of SNPs in a real PGS?

In [ ]:
#@title Answer
cat("A higher PRS means the individual carries more effect alleles, weighted by their GWAS effect sizes.\n")
cat("Most common variants have tiny effects. Summing thousands of SNPs improves predictive power because the signal accumulates \n")
cat("while random noise partially cancels out.\n")

## 3. Load & inspect GWAS summary statistics

We use a published AD GWAS from the GWAS Catalogue:
- Trait: [Alzheimer's disease (MONDO_0004975)](https://www.ebi.ac.uk/gwas/efotraits/MONDO_0004975)
- Study: [GCST007511](https://ftp.ebi.ac.uk/pub/databases/gwas/summary_statistics/GCST007001-GCST008000/GCST007511/)

This file contains, for each variant:
- **SNP**: rsID
- **A1**: effect allele
- **A2**: non-effect allele
- **BETA**: log odds ratio
- **P**: p-value

In [ ]:
# Load genome-wide significant GWAS summary statistics (P<5e-8)
gwas <- fread("gws_ad_gwas.txt")
head(gwas)

In [ ]:
# Basic inspection
cat("Total variants in GWAS:", nrow(gwas), "\n")
cat("Columns:", paste(colnames(gwas), collapse = ", "), "\n")

### Question
- How many genome-wide significant variants are there (P < 5×10⁻⁸)?
- What does the BETA column represent for a binary trait like AD?

In [ ]:
#@title Answer
cat("Genome-wide significant variants:", sum(gwas$P < 5e-8), "\n")
cat("For a binary trait, BETA is the log odds ratio. Exponentiate to get the odds ratio: exp(BETA)\n")

## 4. LD clumping

Nearby SNPs in the genome are often correlated due to **linkage disequilibrium (LD)**, meaning they tend to be inherited together. Including all correlated SNPs in a PGS would mean **counting the same signal multiple times**.

### What does clumping do?
For each genomic region, clumping:
1. Selects the most significant SNP as the **index SNP**
2. Removes all SNPs within a window that are correlated (r² > threshold) with it

### Parameters we use with `plink`
| Parameter | Value | Meaning |
|-----------|-------|---------|
| `--clump-p1` | 1 | Keep all SNPs as potential index SNPs (no p-value filter here) |
| `--clump-r2` | 0.01 | Remove SNPs with r² > 0.01 with the index SNP |
| `--clump-kb` | 1000 | Look within a 1 Mb window |

We use a **European reference panel** to estimate LD (1000 Genomes EUR).

> ⚠️ Clumping is a simplification. More advanced methods (LDpred, PRSice) model LD directly.

In [ ]:
# Try it yourself: run PLINK clumping
# Hint: use system() with intern = TRUE to capture output
# Reference data: eur_ref_prs_workshop
# GWAS file: gws_ad_gwas.txt
# Output folder: output/ (you might need to create it first)
# Output prefix: clumped_ad_gwas

In [ ]:
#@title Answer
ref.data.path <- "eur_ref_prs_workshop"
system("mkdir output/")
cat(system(paste(
  "./plink",
  paste0("--bfile ", ref.data.path),
  "--clump gws_ad_gwas.txt",
  "--clump-p1 1",
  "--clump-r2 0.01",
  "--clump-kb 1000",
  "--out output/clumped_ad_gwas"
), intern = TRUE), sep = "\n")

### Question: How many variants remain after clumping?

In [ ]:
#@title Answer
clumped.gwas <- fread("output/clumped_ad_gwas.clumped")
cat("Variants before clumping:", nrow(gwas), "\n")
cat("Variants after clumping: ", nrow(clumped.gwas), "\n")

## 5. Load & inspect target genotype data

The **target data** is the individual-level genotype dataset that we used in our GWAS and meta-analysis workshops. We will apply the AD PGS on each of the large genetic populations (NEUR and SEUR) and compare the predictive power of the PGS on each group.

The data is in PLINK binary format (`.bed/.bim/.fam`).

In [ ]:
#Load variant information from the genotype data stored in the .bim file
target.bim <- fread("/content/drive/My Drive/VSS_2026/data_GWAS/1kg_phase1_all_EUR_QCed.bim", col.names = c("CHR","SNP","CM","BP","A1","A2"))
head(target.bim)

In [ ]:
#Load sample information from the genotype data stored in the .fam file
target.fam <- fread("/content/drive/My Drive/VSS_2026/data_GWAS/1kg_phase1_all_EUR_QCed.fam")
head(target.fam )

## 6. Harmonisation

Before computing a PGS, we must ensure that the **effect allele in the GWAS** matches the **coded allele in the target genotype data**. If they are misaligned, we would multiply genotypes by the wrong sign, producing an inverted PGS.

### What can go wrong?

| Situation | Problem | Solution |
|-----------|---------|----------|
| Effect allele in GWAS ≠ A1 in target | Beta has the wrong sign | Flip beta sign
| Strand-ambiguous SNPs (A/T or C/G) | Can't tell if strands are flipped | Remove SNPs
| SNPs not in target data | Cannot be used | Remove SNPs

### Strand-ambiguous SNPs
A palindromic SNP (also known as an "ambiguous" SNP) is a SNP in which the possible alleles for that SNP are the same alleles that would pair with each other in the double helix structure. When this occurs, alleles on the forward strand are the same as on the reverse strand (e.g., C/G on forward is G/C on the reverse). See https://mr-dictionary.mrcieu.ac.uk/term/palindrome/ for a visual explanation.

### Steps:
1. Restrict to clumped SNPs
2. Merge GWAS with target BIM
3. Remove strand-ambiguous SNPs
4. Flip BETA where alleles are swapped
5. Exclude SNPs where alleles don't match at all

> **Note**: We only need to do this in R to produce a clean score file. PLINK's `--score` command will handle allele matching during scoring automatically, but the BETA signs must already be correct.

In [ ]:
# ── Step 1: Load target BIM and clumped GWAS ─────────────────────
gwas.clumped  <- gwas[SNP %in% clumped.gwas$SNP]
head(gwas.clumped)

# ── Step 2: Merge target BIM with the clumped GWAS ────────
merged <- merge(target.bim, gwas.clumped[, .(SNP, A1=Effect_allele, A2=Non_Effect_allele, BETA=Beta)],
                by = "SNP", suffixes = c(".target", ".gwas"))
cat("SNPs matched between target and GWAS:", nrow(merged), "\n")

# ── Step 3: Remove strand-ambiguous SNPs ──────────────
ambiguous <- (merged$A1.target == "A" & merged$A2.target == "T") |
             (merged$A1.target == "T" & merged$A2.target == "A") |
             (merged$A1.target == "C" & merged$A2.target == "G") |
             (merged$A1.target == "G" & merged$A2.target == "C")
cat("Strand-ambiguous SNPs removed:", sum(ambiguous), "\n")
merged <- merged[!ambiguous]

# ── Step 4: Flip SNPs where target A1 do not match GWAS A1 ──
same <- merged$A1.target == merged$A1.gwas & merged$A2.target == merged$A2.gwas
flipped <- merged$A1.target == merged$A2.gwas & merged$A2.target == merged$A1.gwas

cat("SNPs with same allele coding:", sum(same), "\n")
cat("SNPs requiring beta flip:", sum(flipped), "\n")

# Drop incompatible SNPs
compatible <- same | flipped
cat("Incompatible SNPs removed:", sum(!compatible), "\n")
merged <- merged[compatible]

# Flip beta where GWAS A1 is not target A1
merged$BETA[flipped[compatible]] <- -merged$BETA[flipped[compatible]]

cat("SNPs retained after harmonisation:", nrow(merged), "\n")

## 7. Create PGS score file

In practice, PLINK's `--score` handles allele matching automatically. But it does need the harmonised, clumped SNPs as input. PLINK's `--score` command reads a simple three-column file:
1. SNP ID
2. Effect allele
3. Effect size (BETA)

In [ ]:
# Create score file from harmonised, clumped SNPs
score <- gwas.clumped[, .(SNP, A1=Effect_allele, BETA=Beta)]
fwrite(score, "output/PGS.score", sep = "\t", col.names = FALSE)
writeLines(score$SNP, "output/score.snps")
cat("Score file written with", nrow(score), "SNPs\n")
head(score)

## 9. Compute PGS per population using PLINK

We now apply the score file to both population A and population B. PLINK multiplies each individual's genotype (0, 1, or 2 copies of the effect allele) by the corresponding BETA, then sums across all SNPs.

We also load phenotype files containing the binary phenotype we are interested on (AD status).


$$\text{PGS}_i = \sum_{j} g_{ij} \times \hat{\beta}_j$$

The `sum` modifier means we get the raw sum (not the mean), which is the standard approach.

In [ ]:
# Load phenotype file
pheno.file <- fread("/content/drive/My Drive/VSS_2026/data_GWAS/Data_1kG_SimuPheno.txt")
pheno.file[, unique(LargePop)]
head(pheno.file)
pheno.file <- pheno.file[, .(IID, Population=LargePop, AD=Y_Binary)]

### Question
- How many individuals are in each population?
- How many AD cases are there per population?

In [ ]:
#@title Answer
cat("Number of individuals per population\n")
pheno.file[, table(Population)]
cat("\nNumber of AD cases per population\n")
pheno.file[AD==1, .N, by=Population]

To apply our derived PGS on the SEUR population, we will use `plink`. Please have look at the `plink` documentation page for the `--score` option (https://www.cog-genomics.org/plink/1.9/score) and try to code it yourself before looking at the answer.

In [ ]:
# Code here

In [ ]:
#@title Answer

# Select only samples from the SEUR population
pop.fam <- target.fam[V2 %in% pheno.file[Population=="SEUR", IID]]
fwrite(pop.fam, "SEUR.fam", sep = "\t", col.names = FALSE)

# Compute PGS for the SEUR population
cat(paste0("=== Population SEUR ===\n"))
plink_cmd <- paste(
  "./plink",
  "--bed", shQuote("/content/drive/My Drive/VSS_2026/data_GWAS/1kg_phase1_all_EUR_QCed.bed"),
  "--fam", shQuote("/content/drive/My Drive/VSS_2026/data_GWAS/1kg_phase1_all_EUR_QCed.fam"),
  "--bim", shQuote("/content/drive/My Drive/VSS_2026/data_GWAS/1kg_phase1_all_EUR_QCed.bim"),
  "--keep", shQuote("SEUR.fam"),
  "--extract", shQuote("output/score.snps"),
  "--score", shQuote("output/PGS.score"), "1 2 3 sum",
  "--out", shQuote("PGS_SEUR")
)

cat(system(plink_cmd, intern=TRUE), sep="\n")


Now run the PGS for the NEUR population:

In [ ]:
# Code here

In [ ]:
#@title Answer

# Select only samples from the NEUR population
pop.fam <- target.fam[V2 %in% pheno.file[Population=="NEUR", IID]]
fwrite(pop.fam, "NEUR.fam", sep = "\t", col.names = FALSE)

# Compute PGS for the NEUR population
cat(paste0("=== Population NEUR ===\n"))
plink_cmd <- paste(
  "./plink",
  "--bed", shQuote("/content/drive/My Drive/VSS_2026/data_GWAS/1kg_phase1_all_EUR_QCed.bed"),
  "--fam", shQuote("/content/drive/My Drive/VSS_2026/data_GWAS/1kg_phase1_all_EUR_QCed.fam"),
  "--bim", shQuote("/content/drive/My Drive/VSS_2026/data_GWAS/1kg_phase1_all_EUR_QCed.bim"),
  "--keep", shQuote("NEUR.fam"),
  "--extract", shQuote("output/score.snps"),
  "--score", shQuote("output/PGS.score"), "1 2 3 sum",
  "--out", shQuote("PGS_NEUR")
)
cat(system(plink_cmd, intern=TRUE), sep="\n")

## 10. Merge PGS with phenotypes

We merge the PLINK-computed PGS with our true phenotype data for **AD** (1 = case, 0 = control)

In [ ]:
# Load PGS output files
pgs_SEUR <- fread("PGS_SEUR.profile")
pgs_NEUR <- fread("PGS_NEUR.profile")
head(pgs_NEUR)

In [ ]:
# Merge PGS with phenotypes
df_SEUR <- merge(pheno.file[Population=="SEUR"], pgs_SEUR[, .(IID, SCORESUM)], by = "IID")
df_NEUR <- merge(pheno.file[Population=="NEUR"], pgs_NEUR[, .(IID, SCORESUM)], by = "IID")
head(df_SEUR)

## 11. PGS evaluation

For a **binary trait** like AD, we evaluate PGS performance using:
- **Logistic regression**: PGS predicting AD case/control status
- **AUC** (Area Under the ROC Curve): ranges from 0.5 (random) to 1.0 (perfect)
- **PGS distribution by case/control status**: cases should have higher PGS on average

In [ ]:
# ── Remove empty predictions ──────────────────────────────
df_NEUR <- na.omit(df_NEUR[, .(IID, AD, SCORESUM)])
df_SEUR <- na.omit(df_SEUR[, .(IID, AD, SCORESUM)])

# ── Fit logistic regression ──────────────────────────────
model_ad_NEUR <- glm(AD ~ SCORESUM, data = df_NEUR, family = binomial)
model_ad_SEUR <- glm(AD ~ SCORESUM, data = df_SEUR, family = binomial)

cat("=== Pop NEUR logistic regression summary ===\n")
print(summary(model_ad_NEUR)$coefficients)

cat("\n\n=== Pop SEUR logistic regression summary ===\n")
print(summary(model_ad_SEUR)$coefficients)

# ── Predicted probabilities ──────────────────────────────
pred_ad_NEUR <- predict(model_ad_NEUR, type = "response")
pred_ad_SEUR <- predict(model_ad_SEUR, type = "response")

In [ ]:
# ── Calculate AUC ────────────────────────────────────────
auc_ad_NEUR <- auc(df_NEUR$AD, pred_ad_NEUR)
auc_ad_SEUR <- auc(df_SEUR$AD, pred_ad_SEUR)

cat("AUC Pop NEUR:", round(auc_ad_NEUR, 3), "\n")
cat("AUC Pop SEUR:", round(auc_ad_SEUR, 3), "\n")

## 13. Cross-population comparison

A key limitation of PGS is that performance often **degrades when applied across ancestries**. This is because:
- LD patterns differ between populations → clumped SNPs may not represent the same signal
- Effect sizes are estimated in one population and may not generalise
- Allele frequencies differ → the same SNP contributes differently to population-level variance

We compare performance for both traits across populations NEUR and SEUR.

In [ ]:
# ── ROC curves ───────────────────────────────────────────
roc_NEUR <- roc(df_NEUR$AD, pred_ad_NEUR)
roc_SEUR <- roc(df_SEUR$AD, pred_ad_SEUR)

par(mfrow = c(1, 2))
plot(roc_NEUR, main = paste0("Pop NEUR — AUC = ", round(auc_ad_NEUR, 3)), col = "#4E79A7", lwd = 2)
plot(roc_SEUR, main = paste0("Pop SEUR — AUC = ", round(auc_ad_SEUR, 3)), col = "#F28E2B", lwd = 2)

In [ ]:
# ── PGS distribution by case/control status ─────────────
df_NEUR$Status <- ifelse(df_NEUR$AD == 1, "Case", "Control")
df_SEUR$Status <- ifelse(df_SEUR$AD == 1, "Case", "Control")
df_NEUR$pop <- "Pop NEUR"
df_SEUR$pop <- "Pop SEUR"

ad_combined <- rbind(df_NEUR, df_SEUR)

ggplot(ad_combined, aes(x = SCORESUM, fill = Status)) +
  geom_density(alpha = 0.45) +
  facet_wrap(~pop) +
  scale_fill_manual(values = c("Case" = "#E15759", "Control" = "#76B7B2")) +
  theme_minimal(base_size = 13) +
  labs(title = "AD PGS Distribution by Case/Control Status",
       x = "Polygenic Risk Score", y = "Density")

### Question
- Is the PGS distribution shifted between cases and controls as expected?
- What does an AUC of 0.774 mean in practice?
- Which population shows better PGS performance, and why might that be?
- What would you do differently if you wanted to improve PGS transferability to Pop NEUR?

In [ ]:
#@title Answer
cat("Cases should have a higher mean PRS than controls, reflecting their higher genetic burden.\n")
cat("An AUC of 0.774 means the PRS correctly ranks a randomly chosen case above a randomly chosen\n")
cat("control 77.4% of the time. For a complex trait like AD this is a meaningful result.\n")

cat("\nPop SEUR likely performs better because the GWAS was based predominantly from Southern European samples.\n")
cat("PGS trained in one populations often underperform in other populations due to:\n")
cat("  1. Different LD structure -> clumped SNPs tag different causal variants\n")
cat("  2. Differences in allele frequencies -> effect size estimates are less accurate\n")
cat("  3. Gene-environment interactions that differ by ancestry\n\n")
cat("To improve transferability:\n")
cat("  - Use population-specific GWAS data to derive effect sizes\n")
cat("  - Use population-specific LD reference panels for clumping\n")
cat("  - Apply methods like PRS-CSx that model ancestry jointly\n")

## 14. Key takeaways

### What we did
1. Downloaded and inspected a published AD GWAS
2. Performed LD clumping to obtain independent signals
3. Harmonised GWAS effect alleles with target genotype data
4. Computed PGS for two populations using PLINK
5. Evaluated PGS performance for a binary (AD) trait
6. Compared performance across populations

### Core concepts

| Concept | Key message |
|---------|-------------|
| **PGS** | Weighted sum of effect alleles across the genome |
| **LD clumping** | Removes redundant correlated SNPs to avoid double-counting |
| **Harmonisation** | Ensures GWAS betas have the correct sign relative to target allele coding |
| **AUC** | Performance metric for binary traits; 0.5 = random, 1.0 = perfect |
| **Cross-population** | PGS trained in one ancestry often underperforms in another |

### Limitations of the C+T approach used today
- Clumping + thresholding (C+T) is simple but suboptimal
- More powerful methods: **PRSice-2**, **LDpred2**, **PRS-CSx** (multi-ancestry)
- PGS does not capture gene-environment interactions
- Rare variants are not included (they need burden test approaches)

### Further reading
- [Choi et al. 2020 — PGS tutorial (Nature Protocols)](https://www.nature.com/articles/s41596-020-0353-1)
- [Martin et al. 2019 — Ancestry and PGS portability](https://www.nature.com/articles/s41588-019-0379-x)
- [LDpred2 paper — Privé et al. 2020](https://academic.oup.com/bioinformatics/article/36/22-23/5424/6039173)